# Challenge 3 — PPO for ALE/MontezumaRevenge-v5
**Group 1 | Machine Learning — Universidad Distrital**

Este notebook entrena el agente PPO y corre el sweep de hiperparámetros.


## 1. Verificar GPU

In [1]:
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('ADVERTENCIA: No hay GPU. Ve a Entorno de ejecución -> Cambiar tipo -> T4 GPU')

GPU disponible: True
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Instalar dependencias

In [2]:
%%capture
!pip install gymnasium[atari] ale-py opencv-python tensorboard autorom
!AutoROM --accept-license

## 3. Clonar repositorio

In [3]:
import os

REPO_URL = 'https://github.com/Johan044/challenge1-dqn-atari'
REPO_DIR = '/content/challenge1-dqn-atari'
CHALLENGE_DIR = os.path.join(REPO_DIR, 'challenge3_group1')

if os.path.exists(REPO_DIR):
    print('Repo ya existe, haciendo pull...')
    !cd {REPO_DIR} && git pull
else:
    print('Clonando repo...')
    !git clone {REPO_URL} {REPO_DIR}

print(f'\nArchivos en challenge3_group1:')
!ls {CHALLENGE_DIR}

Clonando repo...
Cloning into '/content/challenge1-dqn-atari'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (220/220), done.
remote: Total 268 (delta 53), reused 254 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (268/268), 33.58 MiB | 15.66 MiB/s, done.
Resolving deltas: 100% (53/53), done.

Archivos en challenge3_group1:
challenge3_ppo_colab.ipynb  evaluate.py  pyproject.toml      train.py
checklist.md		    model.py	 README.md
env_utils.py		    ppo.py	 sweep_configs.json


## 4. Verificar entorno

In [4]:
import sys
sys.path.insert(0, CHALLENGE_DIR)
os.chdir(CHALLENGE_DIR)

# Sanity check del entorno
!python env_utils.py

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
Observation shape : (4, 84, 84, 1)
Action space      : Discrete(18)
Number of actions : 18
Episode ended at step 464 | total reward: 0.0
env_utils.py OK


## 5. Montar Google Drive (para guardar modelos)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/challenge3_ppo_logs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive montado. Logs se copiarán a: {DRIVE_DIR}')

Mounted at /content/drive
Drive montado. Logs se copiarán a: /content/drive/MyDrive/challenge3_ppo_logs


## 6. Run único — prueba rápida (50k pasos)

In [6]:
# Prueba rápida para verificar que todo funciona antes del sweep completo
# Tarda ~2-3 minutos con GPU
!python train.py --total_steps 51200 --seed 42

2026-05-24 23:11:54.187730: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

Run: lr0.00025_h2048_ep4_bs64_ent0.02_seed42
Device: cuda  |  Config: {'total_steps': 51200, 'horizon': 2048, 'n_epochs': 4, 'batch_size': 64, 'lr': 0.00025, 'gamma': 0.99, 'gae_lambda': 0.95, 'clip_eps': 0.1, 'ent_coef': 0.02, 'vf_coef': 0.5, 'max_grad_norm': 0.5, 'seed': 42}

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
Starting training for 51,200 steps...

step=  20,480 | episodes=   40 | mean_r100=   0.0 | entropy=2.865 | kl=0.0042 | sps=261
step=  40,960 | episodes=   58 | mean_r100=   0.0 | entropy=2.837 | kl=0.0054 | sps=267

Training finished in 0.05h
Best eval mean reward: -inf
Final model saved to : /content/challenge1-dqn-at

## 7. Sweep completo (5 configuraciones × 5M pasos)

In [7]:
# ADVERTENCIA: esto tarda ~5-8 horas en total con GPU T4
# Colab puede desconectarse — si pasa, vuelve a correr esta celda
# Los runs ya completados no se repiten (los checkpoints ya existen)
!python train.py --sweep

2026-05-24 23:16:27.347237: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Running sweep: 5 configurations

Run: sweep_00_lr0.00025_h2048_ep4_bs64_ent0.02_seed0
Device: cuda  |  Config: {'total_steps': 5000000, 'horizon': 2048, 'n_epochs': 4, 'batch_size': 64, 'lr': 0.00025, 'gamma': 0.99, 'gae_lambda': 0.95, 'clip_eps': 0.1, 'ent_coef': 0.02, 'vf_coef': 0.5, 'max_grad_norm': 0.5, 'seed': 0, 'comment': '--- Baseline seed 0 ---'}

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
Starting training for 5,000,000 steps...

step=  20,480 | episodes=   37 | mean_r100=   0.0 | entropy=2.869 | kl=0.0037 | sps=272
step=  40,960 | episodes=   71 | mean_r100=   0.0 | entropy=2.835 | kl=0.0062 | sps=275
step=  61,440 | episode

## 8. Evaluar el mejor modelo

In [ ]:
import glob

# Listar todos los checkpoints disponibles
checkpoints = glob.glob('logs/montezuma_ppo/**/best_model.pt', recursive=True)
print('Checkpoints disponibles:')
for i, ckpt in enumerate(checkpoints):
    print(f'  [{i}] {ckpt}')

In [ ]:
# Cambiar el índice [0] por el checkpoint que quieras evaluar
BEST_CKPT = checkpoints[0]
print(f'Evaluando: {BEST_CKPT}')

!python evaluate.py --checkpoint "{BEST_CKPT}" --n_episodes 10 --save_results

## 9. Ver curvas de aprendizaje con TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/montezuma_ppo

## 10. Copiar logs y modelos a Google Drive

In [ ]:
import shutil

# Copia todos los logs al Drive para no perderlos si Colab se desconecta
src = os.path.join(CHALLENGE_DIR, 'logs')
dst = os.path.join(DRIVE_DIR, 'logs')

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print(f'Logs copiados a Google Drive: {dst}')
print('\nContenido:')
!ls {dst}

## 11. Graficar curvas de aprendizaje (para el paper)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

# Cargar returns de todos los runs
returns_files = glob.glob('logs/montezuma_ppo/**/returns.npy', recursive=True)

fig, ax = plt.subplots(figsize=(10, 5))

for f in returns_files:
    run_name = f.split('montezuma_ppo/')[1].split('/returns')[0]
    returns = np.load(f)

    # Rolling mean 100 episodios
    if len(returns) >= 100:
        rolling = np.convolve(returns, np.ones(100)/100, mode='valid')
        ax.plot(rolling, label=run_name[:40], alpha=0.8)
    else:
        ax.plot(returns, label=run_name[:40], alpha=0.8)

ax.set_xlabel('Episode')
ax.set_ylabel('Mean Return (100-ep window)')
ax.set_title('PPO — ALE/MontezumaRevenge-v5')
ax.legend(fontsize=7, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Guardar figura para el paper
fig_path = os.path.join(DRIVE_DIR, 'learning_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Figura guardada en: {fig_path}')
plt.show()

In [8]:
import threading, shutil, os, time

def auto_backup():
    while True:
        time.sleep(30 * 60)  # cada 30 minutos
        try:
            src = "/content/challenge1-dqn-atari/challenge3_group1/logs"
            dst = "/content/drive/MyDrive/challenge3_ppo_logs/logs"
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"Backup automático guardado: {time.strftime('%H:%M:%S')}")
        except Exception as e:
            print(f"Error en backup: {e}")

t = threading.Thread(target=auto_backup, daemon=True)
t.start()
print("Auto-backup iniciado — cada 30 minutos a Drive")

Auto-backup iniciado — cada 30 minutos a Drive


In [ ]:
import shutil, os

drive_logs = "/content/drive/MyDrive/challenge3_ppo_logs/logs"
local_logs = "/content/challenge1-dqn-atari/challenge3_group1/logs/montezuma_ppo"

if os.path.exists(drive_logs):
    os.makedirs(local_logs, exist_ok=True)
    shutil.copytree(drive_logs, local_logs, dirs_exist_ok=True)
    print("Checkpoints restaurados desde Drive:")
    os.listdir(local_logs)
else:
    print("No hay backup en Drive todavía")